# 🏆 Day 7 — Grand Challenge: Complete Pandas Pipeline
**Goal:** Build a complete data analysis pipeline using ALL functions from the cheatsheet.
**Dataset:** `sales_data.csv`
**Rule:** Try to write code from memory BEFORE checking the cheatsheet!

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('sales_data.csv')
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
print(f"✅ Loaded {len(df)} rows, {len(df.columns)} columns")

## Phase 1: Create & Inspect (from memory!)
Create a manual DataFrame and inspect the sales data.

In [ ]:
# Create a small DF from scratch
summary = pd.DataFrame({
    'Category': ['Electronics', 'Books', 'Clothing'],
    'TotalRevenue': [0, 0, 0],
    'OrderCount': [0, 0, 0]
})
print("=== Manual DataFrame ===")
print(summary)

In [ ]:
# Inspect the sales data
print("=== Inspection ===")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Dtypes:\n{df.dtypes}")
print(f"\nFirst 3 rows:\n{df.head(3)}")
print(f"\nLast 3 rows:\n{df.tail(3)}")
print(f"\nDescribe (numeric):\n{df.describe()}")
print(f"\nDescribe (all):\n{df.describe(include='all')}")

## Phase 2: Clean Data
Handle missing values, duplicates, and convert types.

In [ ]:
# Detect missing data
print("=== Missing Data Detection ===")
print(f"isna() counts:\n{df.isna().sum()}")
print(f"\nnotna() counts:\n{df.notna().sum()}")

In [ ]:
# Handle missing values
print("\n=== Cleaning Missing Data ===")

# Fill Discount with 0 (no discount)
df['Discount'] = df['Discount'].fillna(0)

# Fill Rating with median
df['Rating'] = df['Rating'].fillna(df['Rating'].median())

# Fill City with 'Unknown'
df['City'] = df['City'].fillna('Unknown')

# Drop rows where Status is NaN (pending orders)
df_clean = df.dropna(subset=['Status'])
print(f"Rows after cleaning: {len(df_clean)}")

In [ ]:
# Remove duplicates
df_clean = df_clean.drop_duplicates(subset=['TransactionID'])
print(f"Rows after dedup: {len(df_clean)}")

In [ ]:
# Convert dtypes
df_clean['Rating'] = df_clean['Rating'].astype(int)
print(f"\n=== Updated Dtypes ===")
print(df_clean.dtypes)

## Phase 3: Select & Filter
Use every selection and filtering method.

In [ ]:
# Column selection
print("=== Selection ===")
single = df_clean['Product']  # Series
multi = df_clean[['Product', 'Category', 'TotalAmount']]  # DataFrame
print(f"Single column type: {type(single)}")
print(f"Multi column type: {type(multi)}")

In [ ]:
# .loc[] and .iloc[]
df_loc = df_clean.set_index('TransactionID')
print("\n=== .loc[] ===")
print(df_loc.loc[1, :])

print("\n=== .iloc[] ===")
print(df_clean.iloc[0, :])

In [ ]:
# .at[] and .iat[]
print("\n=== .at[] ===")
print(df_loc.at[1, 'Category'])

print("\n=== .iat[] ===")
print(df_clean.iat[0, 2])

In [ ]:
# Boolean filtering
print("=== Boolean Filters ===")

# Simple filter
expensive = df_clean[df_clean['TotalAmount'] > 500]
print(f"Expensive orders (> $500): {len(expensive)}")

# Multiple conditions
electronics_high = df_clean[(df_clean['Category'] == 'Electronics') & (df_clean['TotalAmount'] > 300)]
print(f"High-value Electronics: {len(electronics_high)}")

# .query()
books_expensive = df_clean.query("Category == 'Books' and TotalAmount > 100")
print(f"Expensive Books: {len(books_expensive)}")

# .isin()
top_cats = df_clean[df_clean['Category'].isin(['Electronics', 'Books'])]
print(f"Electronics + Books: {len(top_cats)}")

# .between()
mid_qty = df_clean[df_clean['Quantity'].between(2, 4)]
print(f"Medium quantity (2-4): {len(mid_qty)}")

## Phase 4: Sort & Modify
Sort data and add/modify columns.

In [ ]:
# Sorting
print("=== Sorting ===")

# Single column ascending
sorted_asc = df_clean.sort_values('TotalAmount')
print(f"Cheapest: {sorted_asc.iloc[0]['Product']} (${sorted_asc.iloc[0]['TotalAmount']})")

# Single column descending
sorted_desc = df_clean.sort_values('TotalAmount', ascending=False)
print(f"Most expensive: {sorted_desc.iloc[0]['Product']} (${sorted_desc.iloc[0]['TotalAmount']})")

# Multiple columns
multi_sorted = df_clean.sort_values(['Category', 'TotalAmount'], ascending=[True, False])
print(f"\nMulti-sorted (Category asc, Amount desc):\n{multi_sorted[['Category', 'Product', 'TotalAmount']].head(6)}")

In [ ]:
# Modify: add columns
print("\n=== Modifications ===")

# Add new column
df_clean['DiscountAmount'] = df_clean['TotalAmount'] * df_clean['Discount']
df_clean['FinalPrice'] = df_clean['TotalAmount'] - df_clean['DiscountAmount']

# .assign() (returns new DF)
df_assigned = df_clean.assign(
    TaxRate=0.08,
    Tax=lambda x: x['TotalAmount'] * x['TaxRate'],
    GrandTotal=lambda x: x['FinalPrice'] + x['Tax']
)
print("=== Using .assign() ===")
print(df_assigned[['Product', 'TotalAmount', 'DiscountAmount', 'FinalPrice', 'Tax', 'GrandTotal']].head())

In [ ]:
# .rename() and .reset_index()
df_renamed = df_assigned.rename(columns={'Product': 'Item', 'GrandTotal': 'FinalCost'})
print(f"\n=== Renamed columns: {df_renamed.columns.tolist()} ===")

## Phase 5: GroupBy & Aggregate
Group data and compute aggregations.

In [ ]:
# GroupBy aggregations
print("=== GroupBy Aggregations ===")

# .groupby().sum()
cat_sum = df_clean.groupby('Category')['TotalAmount'].sum()
print("Revenue by Category:\n", cat_sum)

# .groupby().mean()
cat_mean = df_clean.groupby('Category')['TotalAmount'].mean()
print("\nAvg Sale by Category:\n", cat_mean)

# .groupby().count() and .size()
cat_count = df_clean.groupby('Category')['Product'].count()
print("\nOrder Count by Category:\n", cat_count)

In [ ]:
# .groupby().agg() with dict
print("\n=== Advanced Aggregation ===")
cat_agg = df_clean.groupby('Category').agg({
    'TotalAmount': ['sum', 'mean', 'min', 'max'],
    'Quantity': 'mean',
    'Rating': 'max'
})
print(cat_agg)

In [ ]:
# .pivot_table()
print("\n=== Pivot Table ===")
pivot = df_clean.pivot_table(
    values='TotalAmount',
    index='Category',
    columns='PaymentMethod',
    aggfunc='mean',
    fill_value=0
)
print(pivot)

In [ ]:
# GroupBy → Filter chain
print("\n=== Categories with avg sale > $150 ===")
high_avg = (df_clean.groupby('Category')['TotalAmount']
            .mean()
            .loc[lambda x: x > 150]
            .reset_index())
print(high_avg)

## Phase 6: Merge & Concat
Combine DataFrames.

In [ ]:
# Create two DFs for merging
df_info = pd.DataFrame({
    'Category': ['Electronics', 'Books', 'Clothing'],
    'AvgRating': [4.5, 4.2, 3.8],
    'Growth': ['+15%', '+8%', '-2%']
})

df_stats = pd.DataFrame({
    'Category': ['Electronics', 'Books', 'Clothing'],
    'TotalRevenue': [cat_sum.loc['Electronics'], cat_sum.loc['Books'], cat_sum.loc['Clothing']],
    'OrderCount': [cat_count.loc['Electronics'], cat_count.loc['Books'], cat_count.loc['Clothing']]
})

print("=== df_info ===")
print(df_info)
print("\n=== df_stats ===")
print(df_stats)

In [ ]:
# pd.merge() — all 4 types
print("=== Inner Join ===")
inner = pd.merge(df_info, df_stats, on='Category')
print(inner)

print("\n=== Left Join ===")
left = pd.merge(df_info, df_stats, on='Category', how='left')
print(left)

In [ ]:
# pd.concat() — vertical and horizontal
print("=== Vertical Concat ===")
vertical = pd.concat([df_info, df_stats], ignore_index=True)
print(vertical)

print("\n=== Horizontal Concat ===")
horizontal = pd.concat([df_info, df_stats], axis=1)
print(horizontal)

## Phase 7: Statistics & Math
Compute summary statistics.

In [ ]:
# Basic statistics
print("=== Statistics ===")
print(f"Sum:     ${df_clean['TotalAmount'].sum():,.2f}")
print(f"Mean:    ${df_clean['TotalAmount'].mean():,.2f}")
print(f"Median:  ${df_clean['TotalAmount'].median():,.2f}")
print(f"Min:     ${df_clean['TotalAmount'].min():,.2f}")
print(f"Max:     ${df_clean['TotalAmount'].max():,.2f}")
print(f"Std Dev: ${df_clean['TotalAmount'].std():,.2f}")
print(f"Variance:{df_clean['TotalAmount'].var():,.2f}")
print(f"Count:   {df_clean['TotalAmount'].count()}")

In [ ]:
# Advanced math
df_sorted = df_clean.sort_values('OrderDate')

print("\n=== Cumulative Sum (first 5) ===")
print(df_sorted[['Product', 'TotalAmount']].assign(
    CumSum=df_sorted['TotalAmount'].cumsum()
).head())

print("\n=== Difference (first 5) ===")
print(df_sorted[['Product', 'TotalAmount']].assign(
    Diff=df_sorted['TotalAmount'].diff()
).head())

print("\n=== % Change (first 5) ===")
print(df_sorted[['Product', 'TotalAmount']].assign(
    PctChange=df_sorted['TotalAmount'].pct_change()
).head())

## Phase 8: Strings & Dates
Clean text and extract date info.

In [ ]:
# String operations
print("=== String Operations ===")

# .str.upper()
df_clean['CategoryUpper'] = df_clean['Category'].str.upper()
print(f"Uppercase: {df_clean['CategoryUpper'].head(3).tolist()}")

# .str.lower()
df_clean['ProductLower'] = df_clean['Product'].str.lower()
print(f"Lowercase: {df_clean['ProductLower'].head(3).tolist()}")

# .str.strip()
df_clean['CityClean'] = df_clean['City'].str.strip()

# .str.contains()
laptop_buyers = df_clean[df_clean['Product'].str.contains('Laptop', case=False)]
print(f"\nLaptop buyers: {len(laptop_buyers)}")

# .str.replace()
df_clean['ProductClean'] = df_clean['Product'].str.replace(' ', '_')
print(f"Replaced spaces: {df_clean['ProductClean'].head(3).tolist()}")

# .str.split()
first_names = df_clean['CustomerName'].str.split().str[0]
print(f"First names: {first_names.head(3).tolist()}")

# .str.len()
df_clean['ProductNameLen'] = df_clean['Product'].str.len()
print(f"Name lengths: {df_clean['ProductNameLen'].head(3).tolist()}")

In [ ]:
# Date operations
print("\n=== Date Operations ===")

df_clean['Year'] = df_clean['OrderDate'].dt.year
df_clean['Month'] = df_clean['OrderDate'].dt.month
df_clean['Day'] = df_clean['OrderDate'].dt.day
df_clean['DayOfWeek'] = df_clean['OrderDate'].dt.dayofweek
df_clean['DayName'] = df_clean['OrderDate'].dt.day_name()
df_clean['MonthName'] = df_clean['OrderDate'].dt.month_name()
df_clean['Period'] = df_clean['OrderDate'].dt.to_period('M')

print(df_clean[['Product', 'Year', 'Month', 'Day', 'DayOfWeek', 'DayName', 'MonthName', 'Period']].head())

In [ ]:
# pd.Grouper(freq='D') — daily aggregation
print("\n=== Daily Aggregation (first 5 days) ===")
daily = df_clean.set_index('OrderDate').groupby(pd.Grouper(freq='D')).agg({
    'TotalAmount': ['sum', 'count'],
    'Rating': 'mean'
}).head(5)
print(daily)

## Phase 9: Apply & Deduplicate
Apply custom functions and clean duplicates.

In [ ]:
# .apply() on Series
print("=== Apply Functions ===")
df_clean['DiscountedPrice'] = df_clean['TotalAmount'].apply(lambda x: round(x * 0.9, 2))
print(f"10% off prices (first 3): {df_clean['DiscountedPrice'].head(3).tolist()}")

In [ ]:
# .apply(axis=1) — row-wise
df_clean['Tax'] = df_clean.apply(lambda row: round(row['TotalAmount'] * 0.08, 2), axis=1)
print(f"\nTax (first 3): {df_clean['Tax'].head(3).tolist()}")

In [ ]:
# .applymap() — element-wise on DataFrame
result = df_clean[['Quantity', 'Rating']].applymap(lambda x: round(x * 1.5, 2) if pd.notna(x) else np.nan)
print(f"\nApplyMap (x1.5, first 3):")
print(result.head(3))

In [ ]:
# Deduplication
df_clean = df_clean.drop_duplicates(subset=['TransactionID'])
print(f"\n=== Unique transactions: {len(df_clean)} ===")

In [ ]:
# Value counts
print("\n=== Category Distribution ===")
print(df_clean['Category'].value_counts())
print(f"\nUnique categories: {df_clean['Category'].nunique()}")

## Phase 10: Final Output
Save results to every supported format.

In [ ]:
# Save to all formats
df_clean.to_csv('final_output.csv', index=False)
print("✅ Saved to final_output.csv")

df_clean.to_excel('final_output.xlsx', index=False)
print("✅ Saved to final_output.xlsx")

df_clean.to_json('final_output.json', orient='records', indent=2)
print("✅ Saved to final_output.json")

df_clean.to_parquet('final_output.parquet', index=False)
print("✅ Saved to final_output.parquet")

## 🏆 Final Summary

### Functions Used Today (ALL from cheatsheet):

| Category | Functions |
|----------|----------|
| **Create** | `DataFrame()`, `Series()`, `date_range()`, `to_datetime()` |
| **Read** | `read_csv()`, `read_excel()`, `read_json()` |
| **Inspect** | `head()`, `tail()`, `info()`, `describe()`, `shape`, `columns`, `dtypes`, `index`, `values`, `sample()`, `empty` |
| **Select** | `loc[]`, `iloc[]`, `at[]`, `iat[]`, boolean filter, `query()`, `isin()`, `between()` |
| **Sort** | `sort_values()`, `sort_index()` |
| **Modify** | `drop()`, `rename()`, `set_index()`, `reset_index()`, `assign()`, `astype()` |
| **GroupBy** | `groupby().sum/mean/count/size/first/last`, `agg()`, `pivot_table()` |
| **Merge** | `merge()` (inner/left/right/outer), `concat()` (vertical/horizontal) |
| **Stats** | `sum/mean/median/min/max/std/var/count`, `cumsum/cummax/cummin`, `diff()`, `pct_change()` |
| **Strings** | `.str.upper/lower/strip/contains/replace/split/len/extract` |
| **Dates** | `.dt.year/month/day/hour/minute/second/dayofweek/day_name/month_name/to_period()`, `Grouper()` |
| **NaN** | `isna()/isnull()/notna()/notnull()`, `dropna()`, `fillna()`, `interpolate()` |
| **Apply** | `.apply()`, `.apply(axis=1)`, `.applymap()` |
| **Dedup** | `drop_duplicates()`, `value_counts()`, `nunique()` |
| **Output** | `to_csv/to_excel/to_json/to_parquet` |

### Self-Assessment:
- Rate each section: ✅ Know it / ⚠️ Shaky / ❌ Forgot it
- Fix the ❌ ones by re-running those cells and typing from memory
- Tomorrow: Pick a NEW dataset and do the full pipeline WITHOUT the cheatsheet!